In [ ]:
import numpy as np 
import pandas as pd 

df = pd.read_csv("/kaggle/input/datasets/nicolaswattiez/skillbuilder-data-2009-2010/2012-2013-data-with-predictions-4-final.csv")

df = df[['user_id', 'start_time', 'correct', 'attempt_count', 'ms_first_response']]

df = df.dropna()

df['correct'] = df['correct'].astype(int)
df['attempt_count'] = pd.to_numeric(df['attempt_count'], errors='coerce')
df['ms_first_response'] = pd.to_numeric(df['ms_first_response'], errors='coerce')

df = df.dropna()

In [ ]:
df['start_time'] = pd.to_datetime(df['start_time'], format='mixed', errors='coerce')

In [ ]:
df = df.dropna(subset=['start_time'])

In [ ]:
df = df[['user_id', 'start_time', 'correct', 'attempt_count', 'ms_first_response']]

df = df.dropna()

df['correct'] = df['correct'].astype(int)
df['attempt_count'] = pd.to_numeric(df['attempt_count'], errors='coerce')
df['ms_first_response'] = pd.to_numeric(df['ms_first_response'], errors='coerce')
df['start_time'] = pd.to_datetime(df['start_time'], format='mixed', errors='coerce')

df = df.dropna()
df = df.sort_values(by=['user_id', 'start_time'])

In [ ]:
print(df.shape)
print(df.head())

In [ ]:
df = df.sample(n=200000, random_state=42)
df = df.sort_values(by=['user_id', 'start_time'])

In [ ]:
import numpy as np

window_size = 10
sequences = []

for user_id, group in df.groupby("user_id"):
    group = group.sort_values("start_time")
    
    for i in range(len(group) - window_size):
        window = group.iloc[i:i+window_size]
        
        seq = window[['ms_first_response', 'attempt_count', 'correct']].values
        sequences.append(seq)

In [ ]:
print(len(sequences))
print(sequences[0])

In [ ]:
X = []
y = []

for seq in sequences:
    accuracy = np.mean(seq[:, 2])
    avg_time = np.mean(seq[:, 0])
    avg_attempts = np.mean(seq[:, 1])
    
    X.append([accuracy, avg_time, avg_attempts])
    
    if accuracy < 0.5:
        y.append(1)
    else:
        y.append(0)

In [ ]:
df = df.sample(n=200000, random_state=42)

In [ ]:
print(df.shape)
print(df.head())

In [ ]:
import numpy as np

X = []
y = []

for seq in sequences:
    accuracy = np.mean(seq[:, 2])
    avg_time = np.mean(seq[:, 0])
    avg_attempts = np.mean(seq[:, 1])
    max_attempts = np.max(seq[:, 1])
    time_std = np.std(seq[:, 0])
    correct_std = np.std(seq[:, 2])
    attempt_std = np.std(seq[:, 1])

    X.append([avg_time, avg_attempts, max_attempts, time_std, correct_std, attempt_std])

    if accuracy < 0.5:
        y.append(1)
    else:
        y.append(0)

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)
print(np.bincount(y))







In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced'
)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))

In [ ]:
window_size = 10
future_size = 5

X_sequences = []
y_labels = []

for user_id, group in df.groupby("user_id"):
    group = group.sort_values("start_time")

    if len(group) < window_size + future_size:
        continue

    for i in range(len(group) - window_size - future_size):

        past_window = group.iloc[i:i + window_size]
        future_window = group.iloc[i + window_size:i + window_size + future_size]

        seq = past_window[['ms_first_response', 'attempt_count', 'correct']].values

        correct = seq[:, 2]
        correct_diff = np.diff(correct, prepend=correct[0])

        attempts = seq[:, 1]
        attempt_diff = np.diff(attempts, prepend=attempts[0])

        rolling_correctness = past_window["correct"].rolling(3, min_periods=1).mean().values
        rolling_attempts = past_window["attempt_count"].rolling(3, min_periods=1).mean().values
        relative_time = past_window["ms_first_response"] / past_window["ms_first_response"].mean()
        correct_std = np.full(window_size, np.std(seq[:, 2]))
        time_std = np.full(window_size, np.std(seq[:, 0]))
        attempt_std = np.full(window_size, np.std(seq[:, 1]))
        new_seq = np.column_stack([
            seq[:, 0],             # response time
            seq[:, 1],             # attempts
            seq[:, 2],             # correct
            correct_diff,          # correctness change
            attempt_diff,          # attempt change
            rolling_correctness,   # recent performance trend
            rolling_attempts,      # recent effort trend
            relative_time,          # time compared to student's own window
            correct_std,
            time_std,
            attempt_std
        ])

        #struggling if next 5 questions contain 3 or more wrong answers
        future_correct = future_window['correct'].values
        num_wrong = (future_correct == 0).sum()
        difficulty_score = num_wrong / future_size

        if difficulty_score >= 0.6:
            y = 1
        else:
            y = 0

        X_sequences.append(new_seq)
        y_labels.append(y)

X_seq = np.array(X_sequences)
y_seq = np.array(y_labels)

print(X_seq.shape)
print(y_seq.shape)
print(np.bincount(y_seq))

In [ ]:
print(df['correct'].value_counts(normalize=True))

In [ ]:
print(np.bincount(y_seq))
print(X_seq.shape)
print(y_seq.shape)

In [ ]:
# Normalization of response time 
from sklearn.preprocessing import StandardScaler

X_seq = X_seq.astype("float32")

num_samples, seq_len, num_features = X_seq.shape

scaler = StandardScaler()

# reshape to 2D
X_reshaped = X_seq.reshape(-1, num_features)

# normalize ALL features except "correct" (index 2)
for i in range(num_features):
    if i == 2:  # skip correctness (0/1)
        continue
    X_reshaped[:, i] = scaler.fit_transform(X_reshaped[:, i].reshape(-1, 1)).flatten()

# reshape back
X_seq = X_reshaped.reshape(num_samples, seq_len, num_features)

In [ ]:
#Training and splitting
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, stratify=y_seq, random_state=42
)

In [ ]:
#PyTorch tensors

import torch

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [ ]:
#LSTM

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

#LSTM + ATTENTION MODEL

class LSTMWithAttention(nn.Module):
    def __init__(self, input_size=11, hidden_size=64, num_classes=2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.attention = nn.Linear(hidden_size, 1)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)

        attention_scores = self.attention(lstm_out)
        attention_weights = torch.softmax(attention_scores, dim=1)

        context = torch.sum(attention_weights * lstm_out, dim=1)

        output = self.fc(context)
        return output

In [ ]:
import numpy as np
import torch.nn as nn

class_counts = np.bincount(y_train)
class_weights = len(y_train) / (2 * class_counts)

weights = torch.tensor(class_weights, dtype=torch.float32)

criterion = nn.CrossEntropyLoss()

In [ ]:
#Training LSTM
torch.set_grad_enabled(True)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

attention_model = LSTMWithAttention(input_size=11)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(attention_model.parameters(), lr=0.0005)

epochs = 15

for epoch in range(epochs):
    attention_model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()

        outputs = attention_model(batch_X)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

#Prediction and explanation layer

def predict_probability(attention_model, sequence_tensor):
    """
    Predict probability that one student sequence is struggling.
    sequence_tensor shape should be: (1, 10, 5)
    """
    attention_model.eval()

    with torch.no_grad():
        outputs = attention_model(sequence_tensor)
        probs = F.softmax(outputs, dim=1)
        struggling_prob = probs[:, 1].item()

    return struggling_prob


def get_risk_level(probability):
    if probability >= 0.65:
        return "high"
    elif probability >= 0.45:
        return "medium"
    else:
        return "low"


def generate_reason_codes_from_array(sequence):
    """
    sequence shape: (10, 5)

    Feature order:
    0 = ms_first_response
    1 = attempt_count
    2 = correct
    3 = correctness_change
    4 = attempt_change
    """
    reasons = []

    response_time = sequence[:, 0]
    attempts = sequence[:, 1]
    correctness = sequence[:, 2]

    if correctness[-3:].sum() < correctness[:3].sum():
        reasons.append("DECLINING_CORRECTNESS")

    if attempts[-3:].sum() > attempts[:3].sum():
        reasons.append("INCREASING_ATTEMPTS")

    if response_time[-3:].mean() > response_time.mean() * 1.25:
        reasons.append("SLOW_RESPONSE_TIME")

    if correctness[-3:].sum() == 0:
        reasons.append("RECENT_FAILURE_STREAK")

    return reasons


REASON_TEXT = {
    "DECLINING_CORRECTNESS": "The student's recent correctness is lower than earlier in the sequence.",
    "INCREASING_ATTEMPTS": "The student is needing more attempts in recent interactions.",
    "SLOW_RESPONSE_TIME": "The student is taking longer than usual to respond recently.",
    "RECENT_FAILURE_STREAK": "The student recently answered several questions incorrectly in a row."
}


#INTERVENTION / RECOMMENDATION LAYER


def recommend_intervention(risk_level, reason_codes):
    interventions = []

    if risk_level == "high":
        interventions.append("Immediate teacher review recommended.")

    elif risk_level == "medium":
        interventions.append("Assign targeted practice and monitor closely.")

    else:
        interventions.append("No immediate intervention needed. Continue monitoring.")

    if "DECLINING_CORRECTNESS" in reason_codes:
        interventions.append("Review the recent topic because correctness is declining.")

    if "INCREASING_ATTEMPTS" in reason_codes:
        interventions.append("Provide a worked example because the student needs more attempts.")

    if "SLOW_RESPONSE_TIME" in reason_codes:
        interventions.append("Check for hesitation or confusion; consider giving a hint or simpler warm-up question.")

    if "RECENT_FAILURE_STREAK" in reason_codes:
        interventions.append("Pause progression and give remedial practice before introducing harder questions.")

    return interventions


#TRUST / EVALUATION LAYER

def get_confidence(probability):
    if probability >= 0.75:
        return "very confident struggling"
    elif probability >= 0.50:
        return "confident struggling"
    elif probability >= 0.25:
        return "uncertain / monitor closely"
    else:
        return "low risk confidence"


def evaluate_at_threshold(struggling_probs, y_test_tensor, threshold=0.20):
    predictions = (struggling_probs >= threshold).long()

    print(f"Threshold: {threshold}")
    print("\nClassification Report:")
    print(classification_report(y_test_tensor.numpy(), predictions.numpy()))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test_tensor.numpy(), predictions.numpy()))

    return predictions


def inspect_false_cases(X_test, y_test_tensor, predictions):
    y_true = y_test_tensor.numpy()
    y_pred = predictions.numpy()

    false_negative_indices = np.where((y_true == 1) & (y_pred == 0))[0]
    false_positive_indices = np.where((y_true == 0) & (y_pred == 1))[0]

    if len(false_negative_indices) > 0:
        i = false_negative_indices[0]
        print("\nFalse Negative Example: actual struggling, model missed it")
        print(X_test[i])
    else:
        print("\nNo false negatives found.")

    if len(false_positive_indices) > 0:
        i = false_positive_indices[0]
        print("\nFalse Positive Example: actual not struggling, model raised alert")
        print(X_test[i])
    else:
        print("\nNo false positives found.")


#TEST ONE STUDENT SEQUENCE

student_index = 0

student_sequence_tensor = X_test_tensor[student_index].unsqueeze(0)
student_sequence = X_test[student_index]

probability = predict_probability(attention_model, student_sequence_tensor)
risk_level = get_risk_level(probability)
confidence = get_confidence(probability)

reason_codes = generate_reason_codes_from_array(student_sequence)
explanations = [REASON_TEXT[reason] for reason in reason_codes]
interventions = recommend_intervention(risk_level, reason_codes)

result = {
    "student_id": f"test_student_{student_index}",
    "risk_probability": round(probability, 4),
    "risk_level": risk_level,
    "confidence": confidence,
    "reason_codes": reason_codes,
    "explanations": explanations,
    "recommended_interventions": interventions
}

print(result)


# EVALUATE MODEL AT CHOSEN THRESHOLD

attention_model.eval()

with torch.no_grad():
    outputs = attention_model(X_test_tensor)
    probs = F.softmax(outputs, dim=1)
    struggling_probs = probs[:, 1]

for threshold in [0.45, 0.50, 0.55, 0.60, 0.65]:
    predictions = (struggling_probs >= threshold).long()

    print("\n" + "="*50)
    print(f"Threshold: {threshold}")
    print(classification_report(y_test_tensor.numpy(), predictions.numpy()))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test_tensor.numpy(), predictions.numpy()))

In [ ]:
student_index = 0

student_sequence_tensor = X_test_tensor[student_index].unsqueeze(0)
student_sequence = X_test[student_index]

probability = predict_probability(attention_model, student_sequence_tensor)
risk_level = get_risk_level(probability)
confidence = get_confidence(probability)

reason_codes = generate_reason_codes_from_array(student_sequence)
explanations = [REASON_TEXT[reason] for reason in reason_codes]
interventions = recommend_intervention(risk_level, reason_codes)

result = {
    "student_id": f"test_student_{student_index}",
    "risk_probability": round(probability, 4),
    "risk_level": risk_level,
    "confidence": confidence,
    "reason_codes": reason_codes,
    "explanations": explanations,
    "recommended_interventions": interventions
}

print(result)

In [ ]:
attention_model.eval()

with torch.no_grad():
    outputs = attention_model(X_test_tensor[:5])

print(outputs)
print(outputs.shape)

In [ ]:
print(np.bincount(y_seq))

In [ ]:
def generate_reason_codes(sequence):
    reasons = []

    correctness_values = sequence["correct"]
    attempts = sequence["attempt_count"]
    response_times = sequence["ms_first_response"]

    if correctness_values[-3:].mean() < correctness_values[:3].mean():
        reasons.append("DECLINING_CORRECTNESS")

    if attempts[-3:].mean() > attempts[:3].mean():
        reasons.append("INCREASING_ATTEMPTS")

    if response_times[-3:].mean() > response_times.mean() * 1.25:
        reasons.append("SLOW_RESPONSE_TIME")

    if correctness_values[-3:].sum() == 0:
        reasons.append("RECENT_FAILURE_STREAK")

    return reasons

In [ ]:
def get_risk_level(probability):
    if probability >= 0.50:
        return "high"
    elif probability >= 0.25:
        return "medium"
    else:
        return "low"

In [ ]:
for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
    predictions = (struggling_probs >= threshold).long()

    print("\n" + "="*50)
    print(f"Threshold: {threshold}")
    print(classification_report(y_test_tensor.numpy(), predictions.numpy()))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test_tensor.numpy(), predictions.numpy()))